# Porównanie zdjęcia z kamery vs obraz wczytany jako klatka kamery

Ten notebook porównuje:
- Zdjęcie zrobione bezpośrednio kamerą
- Obraz wczytany z pliku i przekazany jako klatka kamery

In [3]:
import cv2  
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

# Utwórz folder na zdjęcia z kamery
os.makedirs('web-cam-images', exist_ok=True)

# Konfiguracja - rozszerzenie pliku do zapisu
FILE_EXTENSION = '.png'  # Możliwe opcje: '.png', '.jpg', '.jpeg', '.bmp'

# Wybór kamery
CAMERA_INDEX = 1  # 0 - przednia kamera (domyślna), 1 - tylna kamera, 2+ - dodatkowe kamery
# Możliwe wartości:
# 0 - przednia/główna kamera
# 1 - tylna kamera (jeśli dostępna)
# 2, 3, ... - dodatkowe kamery (jeśli dostępne)



## 1. Zrób zdjęcie bezpośrednio z kamery

In [ ]:
# Inicjalizuj kamerę z wybranym indeksem
print(f"Próba otwarcia kamery o indeksie: {CAMERA_INDEX}")
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    print(f"Nie można otworzyć kamery {CAMERA_INDEX}")
    print("Sprawdzanie dostępnych kamer...")
    
    # Sprawdź dostępne kamery (testuj indeksy 0-5)
    available_cameras = []
    for i in range(6):
        test_cap = cv2.VideoCapture(i)
        if test_cap.isOpened():
            available_cameras.append(i)
            test_cap.release()
    
    if available_cameras:
        print(f"Dostępne kamery: {available_cameras}")
        print(f"Spróbuj zmienić CAMERA_INDEX na jeden z tych indeksów.")
    else:
        print("Nie znaleziono żadnych kamer.")
else:
    print(f"✓ Kamera {CAMERA_INDEX} otwarta pomyślnie")
    
    # Pobierz informacje o kamerze
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"Rozdzielczość: {width}x{height}, FPS: {fps}")
    
    # Poczekaj na stabilizację kamery
    print("Stabilizacja kamery...")
    for _ in range(10):
        cap.read()
    
    # Zrób zdjęcie bezpośrednio z kamery - RAW stan matrycy
    ret, direct_camera_image = cap.read()
    
    if ret:
        print(f"\nZdjęcie z kamery (RAW z matrycy): shape={direct_camera_image.shape}, dtype={direct_camera_image.dtype}")
        print(f"Min wartość piksela: {direct_camera_image.min()}, Max: {direct_camera_image.max()}")
        
        # Zapisz w formatach bezstratnych
        # PPM - najprostszy format bez kompresji, "surowy" zapis matrycy
        # ppm_filename = f'web-cam-images/direct_camera_shot_cam{CAMERA_INDEX}.ppm'
        # cv2.imwrite(ppm_filename, direct_camera_image)
        # print(f"\nZapisano RAW (PPM): {ppm_filename}")
        
        # PNG - bezstratny z kompresją (ale zachowuje wszystkie wartości pikseli)
        png_filename = f'web-cam-images/direct_camera_shot_cam{CAMERA_INDEX}.png'
        # Użyj kompresji 0 dla najszybszego zapisu (lub 9 dla najmniejszego pliku)

        # FLAGI
        # https://docs.opencv.org/3.4/d8/d6a/group__imgcodecs__flags.html#ga292d81be8d76901bff7988d18d2b42ac

        cv2.imwrite(png_filename, direct_camera_image, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        print(f"Zapisano RAW (PNG, kompresja=0): {png_filename}")
        
        # Weryfikacja - wczytaj i porównaj
        print("\n=== WERYFIKACJA BEZSTRATNOŚCI ===")
        ppm_loaded = cv2.imread(ppm_filename)
        png_loaded = cv2.imread(png_filename)
        
        ppm_identical = np.array_equal(direct_camera_image, ppm_loaded)
        png_identical = np.array_equal(direct_camera_image, png_loaded)
        
        print(f"PPM identyczny z RAW z kamery: {ppm_identical}")
        print(f"PNG identyczny z RAW z kamery: {png_identical}")
        
        if ppm_identical and png_identical:
            print("✓ Oba formaty zachowują DOKŁADNIE stan matrycy!")
        
    else:
        print("Nie udało się zrobić zdjęcia")
    
    cap.release()
    print(f"\nKamera {CAMERA_INDEX} zwolniona.")



Próba otwarcia kamery o indeksie: 1
✓ Kamera 1 otwarta pomyślnie
Rozdzielczość: 640x480, FPS: 30.0
Stabilizacja kamery...

Zdjęcie z kamery (RAW z matrycy): shape=(480, 640, 3), dtype=uint8
Min wartość piksela: 0, Max: 255

Zapisano RAW (PPM): web-cam-images/direct_camera_shot_cam1.ppm
Zapisano RAW (PNG, kompresja=0): web-cam-images/direct_camera_shot_cam1.png

=== WERYFIKACJA BEZSTRATNOŚCI ===
PPM identyczny z RAW z kamery: True
PNG identyczny z RAW z kamery: True
✓ Oba formaty zachowują DOKŁADNIE stan matrycy!

Kamera 1 zwolniona.

Zdjęcie z kamery (RAW z matrycy): shape=(480, 640, 3), dtype=uint8
Min wartość piksela: 0, Max: 255

Zapisano RAW (PPM): web-cam-images/direct_camera_shot_cam1.ppm
Zapisano RAW (PNG, kompresja=0): web-cam-images/direct_camera_shot_cam1.png

=== WERYFIKACJA BEZSTRATNOŚCI ===
PPM identyczny z RAW z kamery: True
PNG identyczny z RAW z kamery: True
✓ Oba formaty zachowują DOKŁADNIE stan matrycy!

Kamera 1 zwolniona.


## 2. Wczytaj obraz z pliku i przekaż jako klatkę kamery

In [7]:
# Wczytaj zapisane zdjęcie z kamery
filename = f'web-cam-images/direct_camera_shot_cam1{FILE_EXTENSION}'
loaded_image = cv2.imread(filename)

if loaded_image is not None:
    # Konwersja BGR do RGB dla wyświetlania
    loaded_image_rgb = cv2.cvtColor(loaded_image, cv2.COLOR_BGR2RGB)

    # Zapisz dla późniejszego porównania
    loaded_filename = f'web-cam-images/loaded_camera_shot{FILE_EXTENSION}'
    cv2.imwrite(loaded_filename, loaded_image)
    print(f"Wczytany obraz: shape={loaded_image.shape}, dtype={loaded_image.dtype}")
else:
    print("Nie udało się wczytać obrazu")


Wczytany obraz: shape=(480, 640, 3), dtype=uint8


## 3. Porównaj oba obrazy

In [16]:
from index_script import detect_jpeg_compression
import os

def check_image(image_path):
    if not os.path.exists(image_path):
        print(f"Błąd: Plik '{image_path}' nie istnieje!")
        return
    
    print(f"Analiza pliku: {image_path}")
    print("-" * 80)
    
    # try:
    result = detect_jpeg_compression(image_path)
    
    if result is None:
        print("Nie udało się obliczyć wskaźników")
        return
    
    # Wyświetl wskaźniki
    print("\nObliczone wskaźniki:")
    for key, value in result.items():
        if key not in ['filename', 'extension']:
            if isinstance(value, float):
                print(f"{key:20s}: {value:.4f}")
            else:
                print(f"{key:20s}: {value}")
        
    # except Exception as e:
    #     print(f"Błąd: {str(e)}")
    #     import traceback
    #     traceback.print_exc()
    print("-" * 80, "\n")

check_image("web-cam-images/direct_camera_shot_cam1.png")
check_image("test-images/1100x1100-woman.png")

Analiza pliku: web-cam-images/direct_camera_shot_cam1.png
--------------------------------------------------------------------------------

Obliczone wskaźniki:
energy_mean         : 137.6373
energy_std          : 356.0426
estimated_quality   : 95
psnr                : 49.8759
blockiness_mean     : 1.0000
blockiness_max      : 1.0135
blockiness_std      : 0.0097
blockiness_0        : 0.9942
blockiness_1        : 0.9889
blockiness_2        : 1.0135
blockiness_3        : 1.0087
blockiness_4        : 1.0078
blockiness_5        : 0.9933
blockiness_6        : 0.9867
blockiness_7        : 1.0071
-------------------------------------------------------------------------------- 

Analiza pliku: test-images/1100x1100-woman.png
--------------------------------------------------------------------------------

Obliczone wskaźniki:
energy_mean         : 177.2865
energy_std          : 243.4119
estimated_quality   : 95
psnr                : 47.4207
blockiness_mean     : 1.0000
blockiness_max      : 1.

## 4. Wczytanie modelu

In [17]:
import joblib

loaded_model = joblib.load('models/best_model_svm_(rbf).pkl')
loaded_scaler = joblib.load('models/scaler.pkl')
loaded_feature_columns = joblib.load('models/feature_columns.pkl')

print("✓ Model wczytany pomyślnie!")
print(f"✓ Typ modelu: {type(loaded_model).__name__}")
print(f"✓ Liczba cech: {len(loaded_feature_columns)}")

✓ Model wczytany pomyślnie!
✓ Typ modelu: SVC
✓ Liczba cech: 15


## 5. Sprawdzenie obrazów modelem

In [37]:
from index_script import detect_jpeg_compression
import os
import pandas as pd

def check_image(image_path):
    if not os.path.exists(image_path):
        print(f"Błąd: Plik '{image_path}' nie istnieje!")
        return
    
    print(f"Analiza pliku: {image_path}")
    print("-" * 80)
    
    # try:
    result = detect_jpeg_compression(image_path)
    
    if result is None:
        print("Nie udało się obliczyć wskaźników")
        return
    
    result = detect_jpeg_compression(image_path)
    # Wyświetl wskaźniki
    print("\nObliczone wskaźniki:")
    for key, value in result.items():
        if key not in ['filename', 'extension']:
            if isinstance(value, float):
                print(f"{key:20s}: {value:.4f}")
            # else:
            #     print(f"{key:20s}: {value}")
    
    # Przygotuj dane do predykcji
    feature_dict = {col: result.get(col, 0) for col in loaded_feature_columns}
    X_single = pd.DataFrame([feature_dict])
    
    if X_single.isnull().any().any():
        print("\nBrakujące dane w wskaźnikach:")
        print(X_single.isnull().sum()[X_single.isnull().sum() > 0])
        return
    
    # Skalowanie i predykcja
    X_single_scaled = loaded_scaler.transform(X_single)
    prediction = loaded_model.predict(X_single_scaled)[0]
    probabilities = loaded_model.predict_proba(X_single_scaled)[0]
    
    # Wyniki
    print("\n" + "-" * 80)
    print("Wynik predykcji:")
    print("-" * 80)
    print(f"Format: {'PPM (niekompresowany)' if prediction == 1 else 'JFIF (kompresja JPEG)'}")
    print(f"\nPrawdopodobieństwa:")
    print(f"  JFIF:  {probabilities[0]*100:6.2f}%")
    print(f"  PPM:   {probabilities[1]*100:6.2f}%")
    
    confidence = max(probabilities) * 100
    print(f"\nPewność: {confidence:.1f}%")
        
    # except Exception as e:
    #     print(f"Błąd: {str(e)}")
    #     import traceback
    #     traceback.print_exc()
    return 0

In [38]:
check_image("web-cam-images/direct_camera_shot_cam1.png")
check_image("test-images/1100x1100-woman.png")
check_image("web-cam-images/zdjecie_zrobione_przez_aparat(klik).jpg")

Analiza pliku: web-cam-images/direct_camera_shot_cam1.png
--------------------------------------------------------------------------------

Obliczone wskaźniki:
energy_mean         : 137.6373
energy_std          : 356.0426
psnr                : 49.8759
blockiness_mean     : 1.0000
blockiness_max      : 1.0135
blockiness_std      : 0.0097
blockiness_0        : 0.9942
blockiness_1        : 0.9889
blockiness_2        : 1.0135
blockiness_3        : 1.0087
blockiness_4        : 1.0078
blockiness_5        : 0.9933
blockiness_6        : 0.9867
blockiness_7        : 1.0071

--------------------------------------------------------------------------------
Wynik predykcji:
--------------------------------------------------------------------------------
Format: PPM (niekompresowany)

Prawdopodobieństwa:
  JFIF:    0.00%
  PPM:   100.00%

Pewność: 100.0%
Analiza pliku: test-images/1100x1100-woman.png
--------------------------------------------------------------------------------

Obliczone wskaźni

0

# 6. Zdjęcie zrobionę kamerką (ja)

In [61]:
CAMERA_INDEX = 0

# Inicjalizuj kamerę z wybranym indeksem
print(f"Próba otwarcia kamery o indeksie: {CAMERA_INDEX}")
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    print(f"Nie można otworzyć kamery {CAMERA_INDEX}")
else:
    print(f"Kamera {CAMERA_INDEX} otwarta pomyślnie")
    
    # Pobierz informacje o kamerze
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"Rozdzielczość: {width}x{height}, FPS: {fps}")
    
    # Poczekaj na stabilizację kamery
    print("Stabilizacja kamery...")
    for _ in range(10):
        cap.read()
    
    # Zrób zdjęcie bezpośrednio z kamery - RAW stan matrycy
    ret, direct_camera_image = cap.read()
    
    if ret:
        print(f"\nZdjęcie z kamery (RAW z matrycy): shape={direct_camera_image.shape}, dtype={direct_camera_image.dtype}")
        print(f"Min wartość piksela: {direct_camera_image.min()}, Max: {direct_camera_image.max()}")
        
        # PNG - bezstratny z kompresją (ale zachowuje wszystkie wartości pikseli)
        png_filename = f'web-cam-images/direct_camera_shot_cam{CAMERA_INDEX}.bmp'
        # Użyj kompresji 0 dla najszybszego zapisu (lub 9 dla najmniejszego pliku)
        cv2.imwrite(png_filename, direct_camera_image, [cv2.IMWRITE_PNG_COMPRESSION, 0, cv2.IMWRITE_PXM_BINARY, 1])
        print(f"Zapisano RAW (PNG, kompresja=0): {png_filename}")
    else:
        print("Nie udało się zrobić zdjęcia")
    
    cap.release()
    print(f"\nKamera {CAMERA_INDEX} zwolniona.")

Próba otwarcia kamery o indeksie: 0
Kamera 0 otwarta pomyślnie
Rozdzielczość: 640x480, FPS: 30.0
Stabilizacja kamery...
Kamera 0 otwarta pomyślnie
Rozdzielczość: 640x480, FPS: 30.0
Stabilizacja kamery...

Zdjęcie z kamery (RAW z matrycy): shape=(480, 640, 3), dtype=uint8
Min wartość piksela: 0, Max: 255
Zapisano RAW (PNG, kompresja=0): web-cam-images/direct_camera_shot_cam0.bmp

Zdjęcie z kamery (RAW z matrycy): shape=(480, 640, 3), dtype=uint8
Min wartość piksela: 0, Max: 255
Zapisano RAW (PNG, kompresja=0): web-cam-images/direct_camera_shot_cam0.bmp

Kamera 0 zwolniona.

Kamera 0 zwolniona.


In [64]:
check_image("web-cam-images/direct_camera_shot_cam0.bmp")

Analiza pliku: web-cam-images/direct_camera_shot_cam0.bmp
--------------------------------------------------------------------------------

Obliczone wskaźniki:
energy_mean         : 68.7976
energy_std          : 98.7180
psnr                : 63.6712
blockiness_mean     : 1.0012
blockiness_max      : 1.1810
blockiness_std      : 0.1056
blockiness_0        : 1.1810
blockiness_1        : 1.1726
blockiness_2        : 0.9978
blockiness_3        : 0.9640
blockiness_4        : 0.9458
blockiness_5        : 0.9459
blockiness_6        : 0.8988
blockiness_7        : 0.9037

--------------------------------------------------------------------------------
Wynik predykcji:
--------------------------------------------------------------------------------
Format: JFIF (kompresja JPEG)

Prawdopodobieństwa:
  JFIF:   78.18%
  PPM:    21.82%

Pewność: 78.2%

Obliczone wskaźniki:
energy_mean         : 68.7976
energy_std          : 98.7180
psnr                : 63.6712
blockiness_mean     : 1.0012
blockine

0

# Zdjęcie przy użyciu innej biblioteki

In [63]:
import pygame
import pygame.camera
from PIL import Image
import numpy as np

pygame.camera.init()
cameras = pygame.camera.list_cameras()
print(f"Available cameras: {cameras}")

if cameras:
    webcam = pygame.camera.Camera(cameras[0], (640, 480))
    webcam.start()
    
    # Capture image
    img = webcam.get_image()
    
    # Convert to PIL Image
    img_str = pygame.image.tostring(img, 'RGB')
    pil_img = Image.frombytes('RGB', img.get_size(), img_str)
    
    # Save as BMP (uncompressed)
    pil_img.save('web-cam-images/simple_photo.bmp')
    print("Photo saved as BMP")
    
    webcam.stop()

pygame 2.5.2 (SDL 2.28.3, Python 3.10.11)
Hello from the pygame community. https://www.pygame.org/contribute.html
Available cameras: [0, 1]
Available cameras: [0, 1]
Photo saved as BMP
Photo saved as BMP


In [65]:
check_image("web-cam-images/simple_photo.bmp")

Analiza pliku: web-cam-images/simple_photo.bmp
--------------------------------------------------------------------------------

Obliczone wskaźniki:
energy_mean         : 71.8641
energy_std          : 112.2278
psnr                : 62.8691
blockiness_mean     : 1.0013
blockiness_max      : 1.1931
blockiness_std      : 0.1104
blockiness_0        : 1.1815
blockiness_1        : 1.1931
blockiness_2        : 1.0021
blockiness_3        : 0.9293
blockiness_4        : 0.9155
blockiness_5        : 0.9467
blockiness_6        : 0.9236
blockiness_7        : 0.9190

--------------------------------------------------------------------------------
Wynik predykcji:
--------------------------------------------------------------------------------
Format: JFIF (kompresja JPEG)

Prawdopodobieństwa:
  JFIF:   78.18%
  PPM:    21.82%

Pewność: 78.2%

Obliczone wskaźniki:
energy_mean         : 71.8641
energy_std          : 112.2278
psnr                : 62.8691
blockiness_mean     : 1.0013
blockiness_max   

0